encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden? (0:42:47 - 0:57:36)

### Tokenizer'ı eğit (task2'deki BPE)

In [1]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [3]:
def merge(ids, pair, idx):
    # ids içinde art arda gelen pair'leri yeni idx token'ı ile değiştir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [4]:
vocab_size = 1000
num_merges = vocab_size - 256

tokens = list(text.encode("utf-8"))
ids = list(tokens)

merges = {} # (int, int) -> int
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print("byte sayısı :", len(tokens))
print("token sayısı:", len(ids))
print(f"sıkıştırma oranı: {len(tokens) / len(ids):.2f}X")

byte sayısı : 400857
token sayısı: 154399
sıkıştırma oranı: 2.60X


### decode
token id'leri → byte'lar → metin

In [5]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

### encode
Metni byte'lara çevir, sonra `merges`'te **en önce öğrenilmiş** (en küçük idx'li) ikiliden başlayarak birleştir. Birleştirilecek ikili kalmayınca dur.

In [ ]:
def encode(text):
    #en küçük ikiliyi birleştir ve eğer tokenlerimiz arasında hiç yan yana gelmemiş karakter varsa ona inf ver
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

In [ ]:
print(encode("merhaba"))
print(decode(encode("merhaba")))
print(encode("mer"))
print(encode("hab"))
print(encode("a"))
print(encode("Ğ"))
print(encode("ğ"))


[842, 676, 97]
merhaba
[842]
[676]
[97]
[196, 158]
[269]


### decode(encode(metin)) == metin

In [14]:
metin = "Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀 ğışırtı"
ids = encode(metin)
print("metin :", metin)
print("byte sayısı :", len(metin.encode("utf-8")))
print("token sayısı:", len(ids))
print("token'lar:", ids)
print()
print("decode:", decode(ids))
print("aynı mı :", decode(encode(metin)) == metin)

metin : Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀 ğışırtı
byte sayısı : 115
token sayısı: 52
token'lar: [797, 265, 262, 256, 320, 452, 455, 269, 264, 277, 262, 283, 118, 278, 121, 705, 389, 361, 402, 283, 269, 333, 833, 655, 451, 413, 395, 268, 492, 39, 263, 594, 523, 291, 121, 386, 264, 455, 32, 240, 159, 152, 138, 240, 159, 154, 128, 32, 269, 381, 317, 501]

decode: Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀 ğışırtı
aynı mı : True


### Geçerli olmayan UTF-8 dizileri

In [ ]:
# 128 tek başına geçerli bir UTF-8 başlangıç byteı değil 10xxxxxx = devam byte'ı

try:
    print(b"".join(vocab[i] for i in [128]).decode("utf-8"))
except UnicodeDecodeError as e:
    print("strict decode hatası:", e)
# replace kullanırsak bilinmeyen karakter sembolü koyar 
print("errors='replace' ile:", repr(decode([128])))

strict decode hatası: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte
errors='replace' ile: '�'


In [16]:
# 'ş' iki byte: [197, 159] -> sadece ilk byte'ı verirsek
print(list("ş".encode("utf-8")))
print(repr(decode([197])))
print(repr(decode([197, 159])))

# emoji 4 byte; yarısını verirsek
emoji_bytes = list("😊".encode("utf-8"))
print(emoji_bytes)
print(repr(decode(emoji_bytes[:2])))
print(repr(decode(emoji_bytes)))

[197, 159]
'�'
'ş'
[240, 159, 152, 138]
'�'
'😊'


In [ ]:
# tokenları tek tek decode etmek ile hepsini birlikte decode etmek farklı
ids = encode("şöğüç 😊")
print("hepsi birlikte:", decode(ids))
print("tek tek       :", [decode([i]) for i in ids])

hepsi birlikte: şöğüç 😊
tek tek       : ['ş', 'ö', 'ğ', 'üç', ' ', '�', '�', '�', '�']
